In [ ]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset

from medicalDataLoader_Enes import MedicalImageDataset, UnlabeledDataset
from utils_Enes import device, DiceLoss, evaluate, show_predictions, show_confusion
from UNet_Base_Enes import UNet

In [ ]:
FINAL = True
EPOCHS = 100
BATCH_SIZE = 8
LR = 0.001
SEED = 0
EMA_DECAY = 0.99
CONS_WEIGHT = 10
RAMPUP = 30
NOISE_STD = 0.03
if FINAL:
    MODEL_DIR = 'models/mean_teacher_final'
else:
    MODEL_DIR = 'models/mean_teacher'

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

train_set = MedicalImageDataset('train', DATA_DIR, augment=True)
if FINAL:
    train_set = ConcatDataset([train_set, MedicalImageDataset('val', DATA_DIR, augment=True)])
val_set = MedicalImageDataset('val', DATA_DIR)
unlabeled_set = UnlabeledDataset(DATA_DIR)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
unlabeled_loader = DataLoader(unlabeled_set, batch_size=BATCH_SIZE, shuffle=True)

student = UNet(num_classes=4).to(device)
teacher = copy.deepcopy(student)
for p in teacher.parameters():
    p.requires_grad = False

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()
optimizer = torch.optim.Adam(student.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def perturb(x):
    contrast = torch.empty(x.size(0), 1, 1, 1, device=x.device).uniform_(0.8, 1.2)
    brightness = torch.empty(x.size(0), 1, 1, 1, device=x.device).uniform_(-0.1, 0.1)
    noise = torch.randn_like(x) * NOISE_STD
    return (x * contrast + brightness + noise).clamp(0, 1)


def cons_weight(epoch):
    if epoch >= RAMPUP:
        return CONS_WEIGHT
    p = 1 - epoch / RAMPUP
    return CONS_WEIGHT * np.exp(-5 * p * p)


@torch.no_grad()
def update_teacher(step):
    alpha = min(1 - 1 / (step + 1), EMA_DECAY)
    for t, s in zip(teacher.parameters(), student.parameters()):
        t.mul_(alpha).add_(s, alpha=1 - alpha)
    for t, s in zip(teacher.buffers(), student.buffers()):
        t.copy_(s)

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
history = {'train_loss': [], 'cons_loss': [], 'val_dice': [], 'val_hd': [], 'student_dice': []}
best_dice = -1
step = 0

for epoch in range(EPOCHS):
    student.train()
    teacher.train()
    w = cons_weight(epoch)
    losses, cons_losses = [], []
    for (images, targets), u_images in zip(train_loader, unlabeled_loader):
        images, targets, u_images = images.to(device), targets.to(device), u_images.to(device)
        out = student(images)
        sup_loss = ce_loss(out, targets) + dice_loss(out, targets)
        student_probs = torch.softmax(student(perturb(u_images)), dim=1)
        with torch.no_grad():
            teacher_probs = torch.softmax(teacher(perturb(u_images)), dim=1)
        cons_loss = F.mse_loss(student_probs, teacher_probs)
        loss = sup_loss + w * cons_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        update_teacher(step)
        step += 1
        losses.append(sup_loss.item())
        cons_losses.append(cons_loss.item())
    scheduler.step()

    student_dice, _ = evaluate(student, val_loader)
    dice, hd = evaluate(teacher, val_loader)
    history['train_loss'].append(np.mean(losses))
    history['cons_loss'].append(np.mean(cons_losses))
    history['val_dice'].append(dice)
    history['val_hd'].append(hd)
    history['student_dice'].append(student_dice)
    print('Epoch {} | loss {:.4f} cohérence {:.5f} (poids {:.2f}) | prof Dice VD {:.3f} Myo {:.3f} VG {:.3f} | élève {:.3f}'.format(
        epoch, np.mean(losses), np.mean(cons_losses), w, *dice, np.mean(student_dice)))

    if np.mean(dice) > best_dice:
        best_dice = np.mean(dice)
        torch.save(teacher.state_dict(), MODEL_DIR + '/best.pth')

torch.save(teacher.state_dict(), MODEL_DIR + '/last.pth')
np.save(MODEL_DIR + '/history.npy', history)

In [ ]:
plt.plot(np.mean(history['val_dice'], axis=1), label='Professeur')
plt.plot(np.mean(history['student_dice'], axis=1), label='Élève')
plt.xlabel('Époque')
plt.ylabel('Dice moyen (validation)')
plt.legend()
plt.show()

dice, hd = evaluate(teacher, val_loader, post_process=True)
print('Dice VD {:.3f} Myo {:.3f} VG {:.3f} (moy. {:.3f}) | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(*dice, np.mean(dice), *hd))

show_predictions(teacher, val_set, [5, 20, 40, 60], post_process=True)
show_confusion(teacher, val_loader, post_process=True)